In [1]:
#!/home/vanshika/mosaic/bin/python

import argparse
import os
from pathlib import Path
import missionbio.mosaic as ms
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
from matplotlib import lines as mlines
import seaborn as sns
from typing import Tuple, Union
from scipy.stats import binom
from statsmodels.stats.multitest import multipletests
import sys
print("Packages imported.")



/home/AD/vbidhan/mosaic3.12/lib/python3.10/site-packages/missionbio/mosaic/io.py:47: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


Packages imported.


In [2]:
#Logging
# def setup_logging(log_path: str):
#     log_file = open(log_path, 'w')
#     sys.stdout = log_file
#     return log_file

# ===================================
# Helpers for frequency plot
# ===================================

def sort_AP(ids: pd.Series) -> pd.Series:
    """
    Sort order of sample ID's on the plot: Controls followed by patients, and then based on numbers
    """
    k = ids.astype(str).str.extract(r'([A-Za-z]+)(\d+)_([CP])')
    order = (k[2].map({'C': 0, 'P': 1}), k[0].str.upper(), k[1].astype(int))
    order_df = pd.DataFrame({'g': order[0], 'l': order[1], 'n': order[2]})
    return ids.iloc[order_df.sort_values(['g', 'l', 'n']).index]

def plot_variant_freq_by_sample(
    df,
    id_column='ID',
    freq_col='frequency',
    stype_col='sample_type',
    jitter=0.01,
    sample_order=None,
    ax=None,
):
    """
    Scatter plot of variant frequency per sample, coloured by sample_type.
    Returns (fig, ax, samples)
    """
    d = df.copy()

    # Normalize sample_type labels
    d[stype_col] = d[stype_col].replace(
        {'P': 'FTLD-TDP type C', 'C': 'Control'}
    ).astype(str)

    # Determine sample order
    if sample_order is None:
        samples = sort_AP(pd.Series(d[id_column].unique())).tolist()
    else:
        samples = list(sample_order)

    # Palette
    pal = {
        'FTLD-TDP type C': "#F44542"   ,
        'Control': "#92C7FF"
    }
    
    created_ax = False
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 6))
        created_ax = True
    else:
        fig = ax.figure

    seen_labels = set()
    for i, sample in enumerate(samples):
        g = d[d[id_column] == sample]
        if g.empty:
            continue
        x = np.random.normal(loc=i, scale=jitter, size=len(g))
        stype = g[stype_col].iloc[0]
        color = pal.get(stype, 'lightgray')
        label = stype if stype not in seen_labels else None
        if label is not None:
            seen_labels.add(stype)
        ax.scatter(x, g[freq_col].to_numpy(), color=color, s=10, label=label)

    ax.set_xticks(range(len(samples)))
    ax.set_xticklabels(samples, rotation=90, fontsize=9)
    ax.set_xlabel('Sample ID')
    ax.set_ylabel('Variant frequency (%)')

    if seen_labels:
        ax.legend(
            loc='upper left',
            bbox_to_anchor=(1, 1),
            frameon=False,
            title=''
        )

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    fig.tight_layout()

    return fig, ax, samples

def plot_variant_freq_by_sample_gene(
    df,
    id_column='ID',
    freq_col='frequency',
    gene_col='SYMBOL',
    target_genes=None,        
    jitter=0.01,
    sample_order=None,
    ax=None,
    other_color='lightgray'
):
    """
    Scatter plot of variant frequency per sample, coloured by gene.
    Returns (fig, ax, samples).
    """
    if target_genes is None:
        target_genes = []

    d = df.copy()
    d[gene_col] = d[gene_col].astype(str)

    # sample order
    if sample_order is None:
        samples = sort_AP(pd.Series(d[id_column].unique())).tolist()
    else:
        samples = list(sample_order)

    # palette only for target genes present in data
    present_targets = [g for g in target_genes if g in set(d[gene_col])]
    pal = sns.color_palette("tab10", n_colors=max(1, len(present_targets)))
    gene_to_color = {g: pal[i] for i, g in enumerate(present_targets)}

    # figure/axes
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 6))
    else:
        fig = ax.figure

    # plot points per sample with per-point color
    for i, sample in enumerate(samples):
        g = d[d[id_column] == sample]
        if g.empty:
            continue
        x = np.random.normal(loc=i, scale=jitter, size=len(g))
        point_colors = [gene_to_color.get(gene, other_color) for gene in g[gene_col]]
        ax.scatter(x, g[freq_col].to_numpy(), s=10, c=point_colors)

    ax.set_xticks(range(len(samples)))
    ax.set_xticklabels(samples, rotation=90, fontsize=9)
    ax.set_xlabel('Sample ID')
    ax.set_ylabel('Variant frequency (%)')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # legend
    handles = []
    labels = []
    for g in present_targets:
        handles.append(
            mlines.Line2D([0], [0], marker='o', linestyle='', markersize=6,
                          color=gene_to_color[g])
        )
        labels.append(g)

    if (d[gene_col].isin(present_targets) == False).any():
        handles.append(
            mlines.Line2D([0], [0], marker='o', linestyle='', markersize=6,
                          color=other_color)
        )
        labels.append('Other')

    if handles:
        ax.legend(
            handles, labels,
            loc='upper left',
            bbox_to_anchor=(1, 1),
            frameon=False,
            title='Gene'
        )

    fig.tight_layout()
    return fig, ax, samples

def plot_variant_freq_by_sample_callability(
    df,
    id_column='ID',
    freq_col='frequency',
    callability_col='callability',
    jitter=0.01,
    sample_order=None,
    ax=None,
    other_color='lightgray',   # used for NaN / out-of-range
    step=10,                   # 10-point bins: 0–10, 11–20, ...
    cmap_name="viridis"      
):
    """
    Scatter plot of variant frequency per sample, coloured by callability bins.
    Callability expected in [0, 100]. Bins: 0–10, 11–20, ..., 91–100 (by default).

    Returns (fig, ax, samples).
    """
    d = df.copy()
    d[callability_col] = pd.to_numeric(d[callability_col], errors="coerce")
    # sample order
    if sample_order is None:
        samples = sort_AP(pd.Series(d[id_column].unique())).tolist()
    else:
        samples = list(sample_order)

    # Build bins: 0-10, 11-20, ..., 91-100 (inclusive)
    edges = np.arange(0, 100 + step, step)
    def _bin_label(lo, hi):
        if lo == 0:
            return f"{int(lo)}–{int(hi)}"
        return f"{int(lo)+1}–{int(hi)}"

    labels = [_bin_label(edges[i], edges[i+1]) for i in range(len(edges) - 1)]

    d["_call_bin"] = pd.cut(
        d[callability_col],
        bins=edges,
        labels=labels,
        include_lowest=True,
        right=True
    ).astype("object")  # allows NaN to stay NaN


    cmap = plt.get_cmap(cmap_name)
    pal = [cmap(i) for i in np.linspace(0.1, 0.9, len(labels))]
    bin_to_color = dict(zip(labels, pal))

    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 6))
    else:
        fig = ax.figure

    # plot points per sample with per-point color
    for i, sample in enumerate(samples):
        g = d[d[id_column] == sample]
        if g.empty:
            continue
        x = np.random.normal(loc=i, scale=jitter, size=len(g))
        point_colors = [
            bin_to_color.get(b, other_color) if b is not None else other_color
            for b in g["_call_bin"].to_numpy()
        ]
        ax.scatter(x, g[freq_col].to_numpy(), s=10, c=point_colors)

    ax.set_xticks(range(len(samples)))
    ax.set_xticklabels(samples, rotation=90, fontsize=9)
    ax.set_xlabel('Sample ID')
    ax.set_ylabel('Variant frequency (%)')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # legend (all bins that are present)
    present_bins = [lab for lab in labels if (d["_call_bin"] == lab).any()]
    handles, leg_labels = [], []

    for lab in present_bins:
        handles.append(
            mlines.Line2D([0], [0], marker='o', linestyle='', markersize=6,
                          color=bin_to_color[lab])
        )
        leg_labels.append(lab)

    if d["_call_bin"].isna().any():
        handles.append(
            mlines.Line2D([0], [0], marker='o', linestyle='', markersize=6,
                          color=other_color)
        )
        leg_labels.append('Missing/Out of range')

    if handles:
        ax.legend(
            handles, leg_labels,
            loc='upper left',
            bbox_to_anchor=(1, 1),
            frameon=False,
            title=f'Callability ({step}-pt bins)'
        )

    fig.tight_layout()
    return fig, ax, samples

# ====================================
# Compute variant frequency per sample
# ====================================
def compute_variant_frequencies(
    NGT_df: pd.DataFrame,
    n_meta_cols: int = 3,
) -> pd.DataFrame:
    """
    Compute per-sample variant frequency in long format. (n_rows: n_variants x n_samples )

    Parameters
    ----------
    NGT_df : pd.DataFrame
        Genotype dataframe with:
        - Variants as columns 
        - Last `n_meta_cols` columns: at least 'sample_id', 'sample_type', 'ID'
    n_meta_cols : int, default=3
        Number of non-variant columns at the end.

    Returns
    -------
    out : pd.DataFrame
        Long-format table:
        ['sample_id', 'variant_id', 'frequency', 'count',
         'n_cells_callable', 'ID', 'sample_type',
         'n_cells_total', 'callability']
    """
    variant_cols = list(NGT_df.columns[:-n_meta_cols])

    NGT_FILTERED_final = NGT_df.copy()
    sample_ids = NGT_FILTERED_final['sample_id'].unique()

    total_vars = pd.DataFrame(index=sample_ids, columns=variant_cols)
    mut_count = pd.DataFrame(index=sample_ids, columns=variant_cols)
    n_cells_genotyped = pd.DataFrame(index=sample_ids, columns=variant_cols)
    other_variables = pd.DataFrame(
        index=sample_ids,
        columns=['ID', 'sample_type', 'n_cells_total', 'max_freq', 'min_freq']
    )

    for sample in sample_ids:
        sub = NGT_FILTERED_final[NGT_FILTERED_final['sample_id'] == sample]

        sample_type = sub['sample_type'].iat[0]
        ID = sub['ID'].iat[0]
        n_cells = len(sub)

        genotyped_cells = sub[variant_cols].isin([0, 1, 2]).sum()
        counts = sub[variant_cols].isin([1, 2]).sum()
        freqs = counts / genotyped_cells * 100

        total_vars.loc[sample, variant_cols] = freqs
        mut_count.loc[sample, variant_cols] = counts
        n_cells_genotyped.loc[sample, variant_cols] = genotyped_cells

        other_variables.loc[sample, 'sample_type'] = sample_type
        other_variables.loc[sample, 'ID'] = ID
        other_variables.loc[sample, 'n_cells_total'] = n_cells
        other_variables.loc[sample, 'max_freq'] = freqs.max()
        other_variables.loc[sample, 'min_freq'] = freqs.min()

    # Long version of frequencies
    out = (
        total_vars.apply(pd.to_numeric, errors='coerce')
        .stack()
        .rename('frequency')
        .reset_index()
        .rename(columns={'level_0': 'sample_id',
                         'level_1': 'variant_id'})
    )

    # Long counts
    counts_long = (
        mut_count.apply(pd.to_numeric, errors='coerce')
        .stack()
        .rename('count')
        .reset_index()
        .rename(columns={'level_0': 'sample_id',
                         'level_1': 'variant_id'})
    )

    # Long callable cells
    cells_long = (
        n_cells_genotyped.apply(pd.to_numeric, errors='coerce')
        .stack()
        .rename('n_cells_callable')
        .reset_index()
        .rename(columns={'level_0': 'sample_id',
                         'level_1': 'variant_id'})
    )

    # Merge helper columns
    out = (
        out.merge(counts_long, on=['sample_id', 'variant_id'], how='left')
           .merge(cells_long, on=['sample_id', 'variant_id'], how='left')
    )

    ov = other_variables.rename_axis('sample_id').reset_index()

    # Add sample-level metadata
    out = out.merge(
        ov[['sample_id', 'ID', 'sample_type', 'n_cells_total']],
        on='sample_id',
        how='left'
    )

    # Callability
    out['callability'] = out['n_cells_callable'] / out['n_cells_total'] * 100

    return out

def tag_repeats(variants: pd.DataFrame, repeats: pd.DataFrame, bp=5) -> pd.DataFrame:
    """
    Tag variants with whether they fall in/near repeat regions.
    """
    def to_chr(series: pd.Series) -> pd.Series:
        s = series.astype(str).str.strip()
        s = s.str.replace(r'^(chr)?', '', regex=True)  # remove any leading 'chr'
        return 'chr' + s

    v = variants.rename(columns={"position": "pos"}).copy()
    r = repeats.rename(columns={"chrom": "chr"}).copy()

    v["chr"] = to_chr(v["chr"])
    r["chr"] = to_chr(r["chr"])

    v["pos"] = pd.to_numeric(v["pos"], errors="coerce")
    r["start"] = pd.to_numeric(r["start"], errors="coerce")
    r["end"] = pd.to_numeric(r["end"], errors="coerce")

    v = v.dropna(subset=["pos"])
    r = r.dropna(subset=["start", "end"])

    r["start5"] = (r["start"] - bp).clip(lower=1)
    r["end5"] = r["end"] + bp

    v["in_repeat"] = False
    v["near_repeat_5bp"] = False
    for col in ["repClass", "repFamily", "repName"]:
        v[col] = pd.NA

    for cchr in v["chr"].unique():
        v_mask = v["chr"] == cchr
        r_chr = r[r["chr"] == cchr].sort_values("start")
        if r_chr.empty:
            continue

        v_sorted = v.loc[v_mask].sort_values("pos")

        # inside repeat
        m_in = pd.merge_asof(
            v_sorted,
            r_chr[["start", "end", "repClass", "repFamily", "repName"]],
            left_on="pos",
            right_on="start",
            direction="backward",
            allow_exact_matches=True,
            suffixes=("", "_rep")
        ).set_index(v_sorted.index).sort_index()

        in_mask = (m_in["pos"] <= m_in["end"]).fillna(False)
        v.loc[in_mask.index, "in_repeat"] = in_mask.values
        for col in ["repClass", "repFamily", "repName"]:
            v.loc[in_mask.index, col] = m_in[f"{col}_rep"].where(in_mask).values

        # near boundary (±bp), but not inside
        r5 = r_chr[["start5", "end5"]].rename(columns={"start5": "start", "end5": "end"})
        m_near = pd.merge_asof(
            v_sorted,
            r5[["start", "end"]],
            left_on="pos",
            right_on="start",
            direction="backward",
            allow_exact_matches=True
        ).set_index(v_sorted.index).sort_index()

        near_any = (m_near["pos"] <= m_near["end"]).fillna(False)
        v.loc[near_any.index, "near_repeat_5bp"] = (
            near_any & ~v.loc[near_any.index, "in_repeat"]
        ).values

    return v

# ========================================================
# Helpers for binomial thresholding for false positives
# ========================================================

def _materialize_p0_matrix(p0: Union[float, np.ndarray, pd.DataFrame],
                           like_df: pd.DataFrame,
                           shape: Tuple[int, int]) -> np.ndarray:
    if isinstance(p0, (int, float)):
        return np.full(shape, float(p0), dtype=float)
    if isinstance(p0, pd.DataFrame):
        return p0.reindex_like(like_df).to_numpy(dtype=float, copy=False)
    p_mat = np.asarray(p0, dtype=float)
    if p_mat.shape != shape:
        raise ValueError(f"p0 array shape {p_mat.shape} must match {shape}")
    return p_mat

def pvalues_one_sided_less_vec(n: np.ndarray,
                               alt: np.ndarray,
                               het_mask: np.ndarray,
                               p0_mat: np.ndarray) -> np.ndarray:
    """p = P(K <= k | Binom(n, p0)) for het entries; NaN elsewhere."""
    pvals = np.full(n.shape, np.nan, dtype=float)
    rows, cols = np.where(het_mask)
    if rows.size == 0:
        return pvals

    N = n[rows, cols]
    K = alt[rows, cols]
    P = p0_mat[rows, cols]

    valid = (N > 0) & np.isfinite(P)
    if np.any(valid):
        Pv = np.clip(P[valid], 1e-12, 1.0 - 1e-12)
        pvals_sub = binom.cdf(K[valid], N[valid], Pv)
        pvals[rows[valid], cols[valid]] = pvals_sub
    return pvals

def convert_unbalanced_hets(
    NGT: pd.DataFrame,
    DP: pd.DataFrame,
    AF_percent: pd.DataFrame,
    *,
    alpha: float = 0.05,                 # BH-FDR threshold per cell
    p0: Union[float, np.ndarray, pd.DataFrame] = 0.5,
    min_dp: int = 10,                     # ignore hets with DP < min_dp
    apply_only_below_50: bool = True    
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Flip heterozygous calls (NGT==1) to REF (0) if:
      - one-sided binomial p-value (less) BH-FDR q < alpha
    with ALT reconstructed as round((AF%/100) * DP).
    AF_percent is assumed to be in [0,100] (or [1,100]) for sure.

    Returns: (NGT_out, flipped_report, kept_report)
    """
    DP_view = DP.reindex_like(NGT)
    AF_view = AF_percent.reindex_like(NGT)

    NGT_out  = NGT.copy()
    het_mask = (NGT_out.to_numpy(copy=False) == 1)

    # total depth
    dp = np.nan_to_num(DP_view.to_numpy(dtype=float, copy=False), nan=0.0)
    n = np.rint(dp).astype(np.int32, copy=False)
    n[n < 0] = 0

    # Convert AF% to fraction
    afp = np.nan_to_num(AF_view.to_numpy(dtype=float, copy=False), nan=0.0)
    af_frac = np.clip(afp / 100.0, 0.0, 1.0)

    if apply_only_below_50:
        het_mask = het_mask & (af_frac < 0.5)

    alt = np.rint(af_frac * dp).astype(np.int32, copy=False)
    np.clip(alt, 0, None, out=alt)
    np.minimum(alt, n, out=alt)

    if min_dp > 1:
        het_mask = het_mask & (n >= int(min_dp))

    p0_mat = _materialize_p0_matrix(p0, like_df=NGT, shape=n.shape)
    # one-sided p-values
    p_one = pvalues_one_sided_less_vec(n, alt, het_mask, p0_mat)

    # BH-FDR per cell (row-wise) across hets
    p_adj = np.full_like(p_one, np.nan, dtype=float)
    H, W = n.shape
    for i in range(H):
        row_mask = het_mask[i, :]
        if not np.any(row_mask):
            continue
        row_p = p_one[i, row_mask]
        valid = ~np.isnan(row_p)
        if not np.any(valid):
            continue
        _, qvals, *_ = multipletests(row_p[valid], alpha=alpha, method="fdr_bh")
        idx_cols = np.flatnonzero(row_mask)[valid]
        p_adj[i, idx_cols] = qvals

    flip_mask = het_mask & (p_adj < alpha)
    keep_mask = (NGT_out.to_numpy(copy=False) == 1) & ~flip_mask  # keep all other hets

    out_arr = NGT_out.to_numpy(copy=False)
    out_arr[flip_mask] = 0

    def _build_report(mask: np.ndarray, decision: str) -> pd.DataFrame:
        r, c = np.where(mask)
        if r.size == 0:
            return pd.DataFrame(columns=[
                "cell","variant","total_depth","af_percent","alt_reads_est",
                "p0_expected","p_one_sided_less","q_fdr_cell","decision"
            ])
        return pd.DataFrame({
            "cell":             NGT.index.take(r),
            "variant":          NGT.columns.take(c),
            "total_depth":      n[r, c],
            "af_percent":       afp[r, c],
            "alt_reads_est":    alt[r, c],
            "p0_expected":      p0_mat[r, c],
            "p_one_sided_less": p_one[r, c],
            "q_fdr_cell":       p_adj[r, c],
            "decision":         decision
        }).sort_values(["variant", "cell"], ignore_index=True)

    flipped_report = _build_report(flip_mask, "flipped_to_ref")
    kept_report    = _build_report(keep_mask,  "kept_as_het")

    return NGT_out, flipped_report, kept_report

def extract_pos(variant_id):
    return int(variant_id.split(":")[1])

def mark_close_variants_as_missing(geno, variant_ids, amplicon_map, max_distance=25):
    """
    variant_ids: list of variant IDs
    amplicon_map: pd.Series with index=variant_id, value=amplicon_id
    """
    n_cells, n_variants = geno.shape
    positions = np.array([extract_pos(v) for v in variant_ids])
    # Map each variant to an amplicon
    amplicon_ids = amplicon_map.reindex(variant_ids).values
    
    # Precompute amplicon → list of variant indices
    amplicon_to_variants = {}
    for i, amp in enumerate(amplicon_ids):
        amplicon_to_variants.setdefault(amp, []).append(i)
    
    # Modified genotype output
    geno_out = geno.copy()

    # Process each cell
    cell_counter = 0
    variants_counter = 0
    for s in range(n_cells):
        # genotypes for this cell
        row = geno[s, :]
        
        for amp, variant_idx_list in amplicon_to_variants.items():
            idx = np.array(variant_idx_list)

            het_hom = np.where(np.isin(row[idx], [1,2]))[0]
            if len(het_hom) < 2:
                continue

            # Check pairwise distances
            abs_positions = positions[idx][het_hom]
            diff_matrix = np.abs(abs_positions[:, None] - abs_positions[None, :])
            
            # If any pair < max_distance
            if np.any((diff_matrix < max_distance) & (diff_matrix > 0)):
                # Mark *all* those GTs as 3
                geno_out[s, idx[het_hom]] = 3
                cell_counter+=1
                variants_counter += len(het_hom)
    print(f"Total cells affected: {cell_counter}, total variants marked as missing: {variants_counter}")
    return geno_out



In [3]:
print("Loading H5 data...")
data = ms.load("/home/AD/vbidhan/study232-missionbio_TDP-C/data/outputs/integrated_h5_files/hg38/combined_samples_allvars_vafref10_vafhet30_vafhom90.h5", raw=False, filter_variants=False, filter_cells=False, single=True)

NGT_FILTERED = data.dna.get_attribute('NGT_FILTERED')
NGT_raw = data.dna.get_attribute('NGT')
AF_FILTERED = data.dna.get_attribute('AF_FILTERED')
DP = data.dna.get_attribute('DP')
GQ = data.dna.get_attribute('GQ')

# Change Variant Identifier -> Amplicon ID
NGT_FILTERED.columns = data.dna.col_attrs['amplicon']
DP.columns = data.dna.col_attrs['amplicon']
AF_FILTERED.columns = data.dna.col_attrs['amplicon']
GQ.columns = data.dna.col_attrs['amplicon']



Loading H5 data...


            0%: 

In [4]:
# Select the filtered/ final barocodes:
parquet_dir = "/home/AD/vbidhan/study232-missionbio_TDP-C/manuscript_data/parquets/"
NGT_final = pd.read_parquet(f"{parquet_dir}NGT_filtered_STG_final.parquet")

barcodes_to_remove = [barcode for barcode in data.dna.row_attrs['barcode'] if not barcode in NGT_final.index]
print("Barcodes to remove:", len(barcodes_to_remove))
x_amplicons = ["AMPL349615","AMPL349616","AMPL349617","AMPL349618"]
y_amplicons = ["AMPL293806","AMPL293807","AMPL293808","AMPL293809","AMPL293812","AMPL293803","AMPL293804","AMPL293805"]
sex_amplicons = ["AMPL349615","AMPL349616","AMPL349617","AMPL349618","AMPL293806","AMPL293807","AMPL293808","AMPL293809","AMPL293812","AMPL293803","AMPL293804","AMPL293805"]
autosomes = [amplicon for amplicon in data.dna.col_attrs['id'].tolist() if amplicon not in sex_amplicons]
autosome_indices = [
    idx for idx, amplicon in enumerate(data.dna.col_attrs['id']) if amplicon in autosomes
]

gene_names = pd.read_csv('/home/AD/vbidhan/study232-missionbio_TDP-C/design/hg38/amplicons-with-gene-annotation.tsv', sep='\t', header=None)
gene_names.loc[gene_names[3].isin(x_amplicons), 5] = "chrX"
gene_names.loc[gene_names[3].isin(y_amplicons), 5] = "chrY"
mapped_genes = {row[3]: row[5] for _, row in gene_names.iterrows()}
mapped_values = [mapped_genes.get(entry, None) for entry in data.dna.col_attrs['id']]
data.dna.add_col_attr('gene_name', np.array(mapped_values))
selected_amplicons = data.cnv.filter_amplicons(read_depth=10)
print(len(selected_amplicons))



Barcodes to remove: 1564
182


In [5]:
barcodes_to_keep = [barcode for barcode in data.dna.row_attrs['barcode'] if barcode in NGT_final.index]

In [6]:
y_amplicons_filtered = pd.Series(selected_amplicons)[pd.Series(selected_amplicons).isin(y_amplicons)].to_list()
len(y_amplicons_filtered)

data.dna = data.dna.drop(barcodes_to_remove)
# Select the specific barcodes and amplicons for downstream analysis
NGT_FILTERED = NGT_FILTERED.loc[barcodes_to_keep, y_amplicons_filtered]
AF_FILTERED = AF_FILTERED.loc[barcodes_to_keep, y_amplicons_filtered]
DP = DP.loc[barcodes_to_keep, y_amplicons_filtered]
GQ = GQ.loc[barcodes_to_keep, y_amplicons_filtered]



In [7]:
NGT_FILTERED.shape

(39800, 2103)

In [8]:
vars = data.dna.col_attrs['amplicon'].tolist()
idxs = [i for i, c in enumerate(vars) if c in y_amplicons_filtered]
# Fix: index with a NumPy array of positions
ids = data.dna.ids()
selected_vars = np.asarray(ids)[idxs]
selected_vars


array(['chrY:3131889:T/C', 'chrY:3131890:A/G', 'chrY:3131891:T/A', ...,
       'chrY:21853975:T/A', 'chrY:21853975:T/C', 'chrY:21853975:T/G'],
      dtype=object)

In [9]:
male_indices = [i for i, sex in enumerate(data.dna.row_attrs['sex']) if sex == 'M']
male_barcodes = [data.dna.row_attrs['barcode'][i] for i in male_indices]

In [10]:
len(male_barcodes)

27034

Selected male barcodes from the 39,800 neurons. 


In [11]:
NGT_FILTERED_new = NGT_FILTERED.loc[NGT_FILTERED.index.isin(male_barcodes)]
DP_new = DP.loc[DP.index.isin(male_barcodes)]
AF_FILTERED_new = AF_FILTERED.loc[AF_FILTERED.index.isin(male_barcodes)]
GQ_new = GQ.loc[GQ.index.isin(male_barcodes)]

In [12]:
# -------------------------------------------------------
# Filter cells with low average coverage 
# -------------------------------------------------------
print("Filtering cells with good coverage across >80% amplicons")
# DP_collapsed_cleaned = DP_collapsed.drop(columns=amplicons_to_be_dropped)
DP_collapsed = DP_new.T.groupby(DP_new.columns).sum(min_count=1).T
# DP_collapsed_cleaned = DP_collapsed.loc[:, sel_amplicons]

cells_avgcoverage_below10 = pd.DataFrame(
    DP_collapsed.select_dtypes(include='number').lt(10).sum(axis=1)
)
cells_avgcoverage_below10.columns = ['n_amplicons_avgcoverage_below10']
cells_avgcoverage_below10['sample_id'] = data.dna.get_attribute('sample_id')

cells_basedonamplicons = cells_avgcoverage_below10[
    cells_avgcoverage_below10['n_amplicons_avgcoverage_below10']
    >= (0.20 * DP_collapsed.shape[1])
].index

print(f"Cells removed (low coverage >=20% amplicons): {len(cells_basedonamplicons)}")

Filtering cells with good coverage across >80% amplicons
Cells removed (low coverage >=20% amplicons): 1515


In [13]:
# Remove cells
NGT_FILTERED_new = NGT_FILTERED_new.drop(index=cells_basedonamplicons)
DP_new = DP_new.drop(index=cells_basedonamplicons)
AF_FILTERED_new = AF_FILTERED_new.drop(index=cells_basedonamplicons)
GQ_new = GQ_new.drop(index=cells_basedonamplicons)

In [14]:
# -------------------------------------------------------
# Map back to variant IDs
# -------------------------------------------------------
print("Mapping back column names to variant IDs...")
NGT_FILTERED_new.columns = selected_vars
AF_FILTERED_new.columns = selected_vars
DP_new.columns = selected_vars
GQ_new.columns = selected_vars


Mapping back column names to variant IDs...


In [15]:
# ------------------------------------------------------- 
# Remove variant IDs with a spanning deletion allele (marked with asterisk)
# -------------------------------------------------------
masked_vars = [col for col in NGT_FILTERED_new.columns if re.search(r'\*', col)]
print("Variant IDs removed due to spanning deletion:", len(masked_vars))

NGT_FILTERED_new.drop(masked_vars, axis=1, inplace=True)
DP_new.drop(masked_vars, axis=1, inplace=True)
AF_FILTERED_new.drop(masked_vars, axis=1, inplace=True)
GQ_new.drop(masked_vars, axis=1, inplace=True)

Variant IDs removed due to spanning deletion: 9


In [16]:
## Remove variants 
"""
Classify chrY/chrX somatic variant IDs ('chrY:3131889:T/C', GRCh38) for use as a
hemizygous artefact control. Each variant is mapped to its amplicon; each amplicon is
checked for (1) PAR overlap, (2) sequence class (optional BED, e.g. X-transposed /
ampliconic / palindromic / X-degenerate) and (3) empirical homology: the amplicon
insert is re-aligned to the whole genome and the closest off-target copy is reported.
Per variant, a paralogous-sequence-variant (PSV) flag is set when the ALT base equals
the base of the closest homolog at that position, i.e. the 'variant' is most likely a
mismapped read from the homologous locus.

Usage:
  python classify_sex_chrom_variants.py --ids ids.txt --amplicons amplicons.tsv \
      --fasta hg38.fa [--mmi hg38.sr.mmi] [--classes chrY_classes.bed] [--bam merged.bam] \
      -o sexchrom_variant_classes.tsv
"""
import argparse, re
import pandas as pd, pysam, mappy as mp

# GRCh38 pseudoautosomal regions, 1-based inclusive
PAR = {"chrY": [(10_001, 2_781_479), (56_887_903, 57_217_415)],
       "chrX": [(10_001, 2_781_479), (155_701_383, 156_030_895)]}
EXCLUDE_CLASSES = re.compile(r"PAR|X.?transposed|XTR|amplicon|palindrom|heterochrom|centrom|other", re.I)
COMP = str.maketrans("ACGTN", "TGCAN")


def parse_ids(ids):
    df = pd.Series(ids, name="id").str.extract(r"^(?P<chrom>chr[XY]):(?P<pos>\d+):(?P<ref>[ACGTN]+)/(?P<alt>[ACGTN]+)$")
    df["id"], df["pos"] = ids, df["pos"].astype(int)
    return df


def load_intervals(path, name_col):
    """BED-like file (0-based start) -> DataFrame with 1-based inclusive start/end."""
    b = pd.read_csv(path, sep="\t", header=None, comment="#")
    b = b.iloc[:, [0, 1, 2, name_col]]; b.columns = ["chrom", "start", "end", "name"]
    b["start"] += 1
    return b


def overlap(df, iv, label):
    """First interval name containing each variant position (small tables: a merge is fine)."""
    m = df[["id", "chrom", "pos"]].merge(iv, on="chrom")
    m = m[(m.pos >= m.start) & (m.pos <= m.end)].drop_duplicates("id")
    return df.merge(m[["id", "name"]].rename(columns={"name": label}), on="id", how="left")


def homolog_base(hit, qoff, qlen, fa):
    """Reference base aligned to query offset `qoff` in an off-target hit (None if in a gap)."""
    q = qoff if hit.strand == 1 else qlen - 1 - qoff        # mappy reports reverse-strand hits on the revcomp query
    qi, ri = hit.q_st, hit.r_st
    for n, op in hit.cigar:
        if op in (0, 7, 8):                                   # M/=/X consume both
            if qi <= q < qi + n:
                b = fa.fetch(hit.ctg, ri + q - qi, ri + q - qi + 1).upper()
                return b if hit.strand == 1 else b.translate(COMP)
            qi += n; ri += n
        elif op == 1:                                         # I: query only
            if qi <= q < qi + n: return None
            qi += n
        elif op in (2, 3):                                    # D/N: reference only
            ri += n
    return None



In [ ]:
ids = list(NGT_FILTERED_new.columns)
s = pd.Series(ids, name="id")
bad = s[~s.str.match(r"^chr[XY]:\d+:[ACGTN]+/[ACGTN]+$")]
print(len(bad), bad.str.split(":").str[0].value_counts().head(), bad.head(10).tolist(), sep="\n")

0
Series([], Name: id, dtype: int64)
[]


In [18]:
df = s.str.extract(r"^(?P<chrom>chr[^:]+):(?P<pos>\d+):(?P<ref>[^/]+)/(?P<alt>.+)$")
df["id"] = s
print("unparsable:", df.pos.isna().sum())          # should be 0; if not, inspect df[df.pos.isna()]
df = df.dropna(subset=["pos"]).astype({"pos": int})
df = df[df.chrom.isin(["chrX", "chrY"])].reset_index(drop=True)
df.chrom.value_counts()

unparsable: 0


chrY    2094
Name: chrom, dtype: int64

In [19]:
PAR = {"chrY": [(10_001, 2_781_479), (56_887_903, 57_217_415)],
       "chrX": [(10_001, 2_781_479), (155_701_383, 156_030_895)]}
df["in_PAR"] = [any(a <= p <= b for a, b in PAR.get(c, [])) for c, p in zip(df.chrom, df.pos)]
df.in_PAR.value_counts()

False    2094
Name: in_PAR, dtype: int64

In [20]:
amplicons = pd.read_csv("/home/AD/vbidhan/study232-missionbio_TDP-C/design/hg38/amplicons-with-gene-annotation.tsv", sep="\t", header = None)
amplicons.columns = ["chr","start_pos","end_pos","amplicon_id","gene_id","gene_name"]
amp = amplicons.loc[amplicons.amplicon_id.isin(y_amplicons_filtered), ["chr","start_pos","end_pos","amplicon_id","gene_id","gene_name"]].copy()

m = df.merge(amp, left_on="chrom", right_on="chr")
m = m[(m.pos > m.start_pos) & (m.pos <= m.end_pos)]        # assumes BED-style 0-based start
df = df.merge(m[["id","amplicon_id"]], on="id", how="left")
print(amp.assign(length=amp.end_pos - amp.start_pos))
df.amplicon_id.value_counts(dropna=False)

      chr  start_pos   end_pos amplicon_id gene_id gene_name  length
190  chrY    3131888   3132059  AMPL293806       .       NaN     171
192  chrY    5131842   5132061  AMPL293808       .       NaN     219
193  chrY    6131855   6132059  AMPL293809       .       NaN     204
194  chrY    9162269   9162491  AMPL293812       .       NaN     222
195  chrY   16888049  16888251  AMPL293803       .       NaN     202
196  chrY   20838013  20838215  AMPL293804       .       NaN     202
197  chrY   21853753  21853975  AMPL293805       .       NaN     222


AMPL293804    322
AMPL293808    308
AMPL293805    302
AMPL293812    301
AMPL293803    299
AMPL293809    286
AMPL293806    276
Name: amplicon_id, dtype: int64

In [21]:
import pysam, mappy as mp
fa = pysam.FastaFile("/complgen/refseq/hg38/genome_hg38.fa")                       # contig names must match ('chrY' vs 'Y')
al = mp.Aligner("/complgen/refseq/hg38/genome_hg38.fa", preset="sr", best_n=50)    # indexing takes several minutes; save with fn_idx_out="hg38.sr.mmi" and reuse

res = []
for r in amp.itertuples():
    seq = fa.fetch(r.chr, r.start_pos, r.end_pos).upper()
    off = [h for h in al.map(seq) if not (h.ctg == r.chr and h.r_st < r.end_pos and h.r_en > r.start_pos)]
    b = max(off, key=lambda h: h.mlen / h.blen * (h.q_en - h.q_st), default=None)
    res.append(dict(amplicon_id=r.amplicon_id, length=len(seq), n_offtarget=len(off),
                    best_hit=f"{b.ctg}:{b.r_st+1}-{b.r_en}" if b else None,
                    identity=round(b.mlen / b.blen, 3) if b else 0,
                    query_cov=round((b.q_en - b.q_st) / len(seq), 2) if b else 0,
                    mismatches=b.NM if b else None))
hom = pd.DataFrame(res); hom

,amplicon_id,length,n_offtarget,best_hit,identity,query_cov,mismatches
0,AMPL293806,171,1,chrX:89475699-89475869,0.988,1.0,2.0
1,AMPL293808,219,1,chrX:91909770-91909988,0.977,1.0,5.0
2,AMPL293809,204,1,chrX:93015589-93015792,0.980,1.0,4.0
3,AMPL293812,222,1,chrY:21701821-21702042,0.986,1.0,3.0
4,AMPL293803,202,0,None,0.000,0.0,NaN
5,AMPL293804,202,0,None,0.000,0.0,NaN
6,AMPL293805,222,0,None,0.000,0.0,NaN


In [22]:
ref_ok = [fa.fetch(c, p - 1, p - 1 + len(r)).upper() == r for c, p, r in zip(df.chrom, df.pos, df.ref)]
print(f"REF matches FASTA: {sum(ref_ok)}/{len(ref_ok)}")

for r in amp.itertuples():                                         # each insert should find itself at 100%
    self_hit = [h for h in al.map(fa.fetch(r.chr, r.start_pos, r.end_pos))
                if h.ctg == r.chr and h.r_st < r.end_pos and h.r_en > r.start_pos]
    print(r.amplicon_id, [(round(h.mlen / h.blen, 3), h.mapq) for h in self_hit])

REF matches FASTA: 2094/2094
AMPL293806 [(1.0, 45)]
AMPL293808 [(1.0, 60)]
AMPL293809 [(1.0, 60)]
AMPL293812 [(1.0, 50)]
AMPL293803 [(1.0, 60)]
AMPL293804 [(1.0, 60)]
AMPL293805 [(1.0, 60)]


In [23]:
usable = ["AMPL293803", "AMPL293804", "AMPL293805"]
ids3 = df.loc[df.amplicon_id.isin(usable), "id"].tolist()

NGT_FILTERED_new = NGT_FILTERED_new.loc[:, ids3].copy()
AF_FILTERED_new  = AF_FILTERED_new.loc[:, ids3].copy()
DP_new  = DP_new.loc[:, ids3].copy()
GQ_new  = GQ_new.loc[:, ids3].copy()


In [84]:
NGT_FILTERED_new.shape

(23444, 1234)

In [27]:
# -------------------------------------------------------
# Attach metadata
# -------------------------------------------------------
print("Building metadata table...")
#supply empty list (trail)
cells_basedonamplicons = []
NGT_FILTERED_new['ID'] = data.dna.get_attribute('ID').drop(index=cells_basedonamplicons)
NGT_FILTERED_new['sample_id'] = data.dna.get_attribute('sample_id').drop(index=cells_basedonamplicons)
NGT_FILTERED_new['sample_type'] = data.dna.get_attribute('sample_type').drop(index=cells_basedonamplicons)

metadata_filtered = pd.DataFrame()
metadata_filtered['barcode'] = data.dna.get_attribute('barcode').drop(index=cells_basedonamplicons)
metadata_filtered['ID'] = data.dna.get_attribute('ID').drop(index=cells_basedonamplicons)
metadata_filtered['age_at_death'] = data.dna.get_attribute('age_at_death').drop(index=cells_basedonamplicons)
metadata_filtered['age_at_onset'] = data.dna.get_attribute('age_at_onset').drop(index=cells_basedonamplicons)
metadata_filtered['sample_id'] = data.dna.get_attribute('sample_id').drop(index=cells_basedonamplicons)
metadata_filtered['sex'] = data.dna.get_attribute('sex').drop(index=cells_basedonamplicons)
metadata_filtered['sample_type'] = data.dna.get_attribute('sample_type').drop(index=cells_basedonamplicons)
metadata_filtered['hemisphere'] = data.dna.get_attribute('hemisphere_ab').drop(index=cells_basedonamplicons)
metadata_filtered['PMI'] = data.dna.get_attribute('PMI').drop(index=cells_basedonamplicons)
metadata_filtered['doublet_score'] = data.dna.get_attribute('doublet_score').drop(index=cells_basedonamplicons)
metadata_filtered['ethnicity'] = data.dna.get_attribute('ethnicity').drop(index=cells_basedonamplicons)
metadata_filtered['sample_tissue'] = data.dna.get_attribute('sample_tissue').drop(index=cells_basedonamplicons)
metadata_filtered['site'] = data.dna.get_attribute('site').drop(index=cells_basedonamplicons)
metadata_filtered['pool'] = data.dna.get_attribute('pool').drop(index=cells_basedonamplicons)
metadata_filtered['race'] = data.dna.get_attribute('race').drop(index=cells_basedonamplicons)
metadata_filtered.set_index('barcode', inplace=True)

Building metadata table...


In [28]:
# -------------------------------------------------------
# Apply germline and callability mask to the NGT dataframe 
# -------------------------------------------------------
print("Converting variants with a freq >= 65 and callability < 70 in a sample to missing")
out = compute_variant_frequencies(NGT_FILTERED_new)
out_filtered = out[out['frequency'] > 0].copy()
conditions = out_filtered[
    (out_filtered['frequency'] >= 65) | (out_filtered['callability'] < 70)
]

for _, row in conditions.iterrows():
    sample_id = row['sample_id']
    variant_id = row['variant_id']
    if variant_id not in NGT_FILTERED_new.columns:
        continue
    mask = NGT_FILTERED_new['sample_id'].eq(sample_id)
    NGT_FILTERED_new.loc[mask, variant_id] = 3

vars_invalid = (~NGT_FILTERED_new.iloc[:, :-3].isin([1, 2]).any())
vars_invalid = (vars_invalid[vars_invalid==True].index.to_list())
print("Variants entirely removed due to poor callability / germline detection:", len(vars_invalid))

Converting variants with a freq >= 65 and callability < 70 in a sample to missing
Variants entirely removed due to poor callability / germline detection: 17


In [29]:
NGT_FILTERED_new = NGT_FILTERED_new.iloc[:, :-3] 
if vars_invalid:
    NGT_FILTERED_new.drop(vars_invalid,axis=1, inplace=True)
    DP_new.drop(vars_invalid, axis=1 ,inplace=True)
    AF_FILTERED_new.drop(vars_invalid, axis=1, inplace=True)
    GQ_new.drop(vars_invalid, axis=1, inplace=True)



In [30]:
df[df['id'].isin(NGT_FILTERED_new.columns)]['pos'].nunique()

555

In [ ]:
## For the 142 positive calls, calculate the sample overlap
ids = metadata_filtered['ID'].reindex(NGT_FILTERED_new.index)
print(f"{ids.isna().sum()} barcodes not found in metadata_filtered (excluded)")

# True if ≥1 barcode of that patient has genotype 2 → one count per patient per variant
n_pat = NGT_FILTERED_new.eq(2).groupby(ids).any().sum()

# Distribution: how many variants are seen in 1, 2, 3, … patients
dist = n_pat[n_pat > 0].value_counts().sort_index()
for k, v in dist.items():
    print(f"{v} variants observed in {k} patient(s)")

0 barcodes not found in metadata_filtered (excluded)
105 variants observed in 1 patient(s)
11 variants observed in 2 patient(s)
4 variants observed in 3 patient(s)


In [32]:
print(NGT_FILTERED_new.shape)
x = np.array(NGT_FILTERED_new)
value_counts = pd.Series(x.ravel()).value_counts().sort_index()
print(value_counts)

(25519, 906)
0    19185767
1        6102
2         139
3     3928206
dtype: int64


In [34]:
6102 / (19185767 + 6102 + 139) * 100

0.03179448445415404

In [33]:
y = np.array(NGT_raw.loc[(NGT_FILTERED_new.index), (NGT_FILTERED_new.columns)])
print(y.shape)
value_counts = pd.Series(y.ravel()).value_counts().sort_index()
print(value_counts)

(25519, 906)
0    22213257
1       38513
2         427
3      868017
dtype: int64


In [35]:
38513 / (22213257 + 38513 + 427) * 100

0.17307504512925173